In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Set the style to colorblind-friendly
plt.style.use('seaborn-v0_8-colorblind') # Use 'seaborn-colorblind' for older mpl

## Curvas 1

In [ ]:

# Caminho da pasta com os CSVs
pasta = "./tests/results/mylib"

# Listas para médias
rows = []

In [ ]:
# Ler todos os CSV da pasta
for arquivo in os.listdir(pasta):
    if arquivo.endswith('.csv'):
        parts = arquivo.split('-')
        broadphase = parts[0]
        collision = parts[1]
        count = parts[2]

        if count.isnumeric() == False:
            continue

        caminho = os.path.join(pasta, arquivo)

        df = pd.read_csv(caminho)
        rows.append([
            df["collisionsTest"].mean(), 
            df["broadphaseTime"].mean(),
            df["narrowphaseTime"].mean(),
            df["integrationTime"].mean(),
            df["relaxationTime"].mean(),
            df["separationTime"].mean(),
            df["gridClearTime"].mean(),
            df["gridInitTime"].mean(),
            df["gridSortTime"].mean(),
            df["deltatime"].mean(), 
            int(count), 
            broadphase, 
            collision
        ])

In [ ]:
df1 = pd.DataFrame(
    rows, 
    columns=[
        "collisions_test", 
        "broadphase_time",
        "narrowphase_time",
        "integration_time",
        "relaxation_time",
        "separation_time",
        "gridClear_time",
        "gridInit_time",
        "gridSort_time",
        "dt", 
        "count", 
        "broadphase", 
        "narrowphase"
        ]
    )


In [ ]:
styles = {
    ("naive", "sat"): {"linestyle": "-", "marker": "o"},
    ("grid", "sat"): {"linestyle": "-", "marker": "o"},
    ("naive", "gjk"): {"linestyle": "--", "marker": "s"},
    ("grid", "gjk"): {"linestyle": "--", "marker": "s"},
}

In [ ]:
fig, ax = plt.subplots()

for broadphase in ["naive", "grid"]:
    g = df1[
        (df1["broadphase"] == broadphase) & 
        (df1["narrowphase"] == "sat")
    ].sort_values("count")


    if broadphase == "grid":
        print(g[["count", "collisions_test"]])

    style = styles[(broadphase, "sat")]

    ax.plot(
        g["count"],
        g["collisions_test"],
        label=broadphase,
        linestyle=style["linestyle"],
        marker=style["marker"]
    )

ax.set_xlabel("Nº de objetos")
ax.set_ylabel("Média de testes de colisão")
ax.set_yscale("log")
ax.legend()
ax.grid()

plt.show()

## Curvas 2

In [ ]:

# Caminho da pasta com os CSVs
pasta = "./tests/results/box2d"

# Listas para médias
rows = []

In [ ]:
# Ler todos os CSV da pasta
for arquivo in os.listdir(pasta):
    if arquivo.endswith('.csv'):
        parts = arquivo.split('-')
        count = parts[1]
        if count.isnumeric() is False:
            continue

        caminho = os.path.join(pasta, arquivo)

        df = pd.read_csv(caminho)
        rows.append([
            # df["true_collisions"].mean(), 
            df["dt"].mean(), 
            int(count)
        ])

In [ ]:
df2 = pd.DataFrame(rows, columns=["dt", "count"])
df2 = df2.sort_values("count")

In [ ]:
fig, ax = plt.subplots()

for narrowphase in ["sat", "gjk"]:
    g = df1[
            (df1["narrowphase"] == narrowphase) & 
            (df1["broadphase"] == "grid")
        ].sort_values("count")
    
    style = styles[("grid", narrowphase)]
    ax.plot(
        g["count"],
        g["dt"],
        label=narrowphase,
        linestyle=style["linestyle"],
        marker=style["marker"]
    )

ax.plot(df2["count"], df2["dt"], label="planckjs", marker="o")

ax.set_xlabel("Nº de objetos")
ax.set_ylabel("Tempo médio (ms)")
# ax.set_yscale("log")
ax.legend()
ax.grid()

plt.show()

## Curvas 3

In [ ]:
cols = [
    "narrowphase_time",
    "broadphase_time",
    "relaxation_time",
    "integration_time",
    "separation_time",
    "gridInit_time",
    "gridSort_time",
    "gridClear_time",
]

# padrões (hatches) distintos
hatches = ['//', '\\\\', 'xx', '--', '++', 'oo', '..', '**']

for narrowphase in ["sat", "gjk"]:
    g = df1[
        (df1["broadphase"] == "grid") & 
        (df1["narrowphase"] == narrowphase)
    ].sort_values("count")

    # calcula porcentagens de uma vez
    rel = g[cols].div(g["dt"], axis=0) * 100

    x = g["count"].values
    y = rel.values.T  # necessário para stackplot

    fig, ax = plt.subplots(figsize=(10, 6))

    # gráfico empilhado manual (permite mais controle)
    stacks = ax.stackplot(x, y, edgecolor='black')

    # aplica hatches
    for poly, hatch in zip(stacks, hatches):
        poly.set_hatch(hatch)
        poly.set_alpha(0.9)

    ax.legend(stacks, cols)

    ax.set_xlabel("Nº de objetos")
    ax.set_ylabel("% do tempo total")

    # plt.tight_layout()
    plt.show()